In [ ]:
import os

import sys
import time

import contextily as ctx
import geopandas as gpd
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from datetime import date, datetime
from zoneinfo import ZoneInfo

from matplotlib import colors

sys.path.insert(1, '../../narrow_rivers_PIXC/scripts/')
from reaches import *
from utils import *

### NHD Waterbody

In [ ]:
index = 4

data_path = '/nas/cee-ice/data/NHDPlus_HR/'

## Set-up
mdata_path = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC/data/'
# Max binsize of 1000 is plenty for CONUS w/o lakes
# Mississippi is ~472 m wide at mouth as calculated here
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 150, 200, 500, 1000]

In [ ]:
# Define dtypes for lookup tables to preserve leading zeros
dtype_dic= {'HUC4': str, 'HUC2': str, 'toBasin': str, 'level': str}
# Read in HUC lookup table
lookup = pd.read_csv(os.path.join(mdata_path,
                                  'HUC4_lookup_no_great_lakes.csv'),
                     dtype=dtype_dic)

# Get current HUC2 and HUC4 IDs
huc2 = f'HUC2_{lookup.loc[index,'HUC4'][0:2]}'
huc4 = f'NHDPLUS_H_{lookup.loc[index,'HUC4']}_HU4_GDB'

# Set data filepath
file_path = os.path.join(data_path, huc2, huc4, huc4 + '.gdb')

In [ ]:
area = gpd.read_file(filename=file_path, layer='NHDArea',
                      engine='pyogrio')

### Functions

In [ ]:
def get_PIXC_files(filepath, tiles):
    files = []

    for file in os.listdir(filepath):
        for tile in tiles:
            if tile in file:
                files.append(os.path.join(filepath, file))

    return files

In [ ]:
def read_in_masks(site_id):
    masks = gpd.read_parquet('/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/nhd_reach_masks_for_gauges_combined.parquet')
    reach_mask = masks[masks['site_id'] == site_id].set_crs(epsg=3857)
    buffered_mask = reach_mask.buffer(300, cap_style='flat')
    
    temp = gpd.GeoSeries(buffered_mask).to_crs('EPSG:4326')
    bounds = temp.total_bounds
    del temp

    print()
    huc2 = masks[masks['site_id'] == site_id]['huc2'].iloc[0]

    return reach_mask, buffered_mask, bounds, huc2

In [ ]:
def bitwiseMask(ds, level):
    '''
    This function masks a PIXC granule: for now, it ony remove pixels
    with land classification and those with bad geolocation_qual.
    # See page 65 of PIXC PDD: https://podaac.jpl.nasa.gov/SWOT?tab=datasets-information&sections=about%2Bdata
    '''
    # Fow now, eliminate the really bad stuff
    # mask = np.where((ds.classification > 1) & (ds.geolocation_qual < 2**16) &
    #                 (np.abs(ds.cross_track) > 10000) & (np.abs(ds.cross_track) < 60000))[0]


    mask = np.where((ds.classification > 2) &
            (ds.interferogram_qual < 2**level) &
            (ds.classification_qual < 2**level) &
            (ds.geolocation_qual < 2**level) &
            (ds.sig0_qual < 2**level) &
            (np.abs(ds.cross_track) > 10000) &
            (np.abs(ds.cross_track) < 60000))[0]
    
    print(mask.shape)
    return mask

In [ ]:
def find_SWOT_granules(site_id):
    nhd_gauges = gpd.read_parquet('/nas/cee-water/cjgleason/fiona/USGS_gauges/usgs_stations_with_stage_and_nhd_metadata.parquet')

    dtype_dic= {'tile': str, 'site_id': str}
    lookup = pd.read_csv('/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC/data/usgs_gauges_swot_science_tiles.csv',
                         dtype=dtype_dic)
    tiles = lookup[lookup['site_id'] == site_id]['tile'].to_list()
    print(tiles)

    data_dir = f'/nas/cee-ice/data/SWOT/PIXC_vD'
    print(data_dir)

    # Get metadata needed later
    lat = nhd_gauges[nhd_gauges['site_id'] == site_id]['dec_lat_va'].values[0]
    long = nhd_gauges[nhd_gauges['site_id'] == site_id]['dec_long_va'].values[0]
    alt = nhd_gauges[nhd_gauges['site_id'] == site_id]['alt_va'].values[0] * 0.3048
    # No need to extract horizontal datum--dec_coord_datum_cd always equals NAD83
    vert_datum = nhd_gauges[nhd_gauges['site_id'] == site_id]['alt_datum_cd'].values[0]

    PIXC_files = get_PIXC_files(filepath=data_dir, tiles=tiles)
    return PIXC_files, lat, long, alt, vert_datum

In [ ]:
# Get metadata
def get_metadata(pixc_files, index):
    pixc_path = pixc_files[index]
    # print(pixc_path)
    granule_name = pixc_path[31:-3]
    # print(f'Granule name: {granule_name}')
    pass_num = int(granule_name[20:23])
    # print(f'Pass number: {pass_num}')
    pixc_timestamp = granule_name[29:44]
    # print(f'Timestamp: {pixc_timestamp}')
    pixc_dt = datetime.strptime(pixc_timestamp, '%Y%m%dT%H%M%S')
    pixc_dt = pixc_dt.replace(tzinfo=ZoneInfo('UTC'))

    return pixc_path, granule_name, pass_num, pixc_dt

In [ ]:
# Read in PIXC and mask
def read_PIXC_and_process(pixc_path, bounds, level):
    # Read in pixel group
    ds_PIXC = xr.open_mfdataset(paths=pixc_path, group = 'pixel_cloud',
    
                                engine='h5netcdf')
    lon = ds_PIXC.longitude.compute()
    lat = ds_PIXC.latitude.compute()
    
    ds_subset = ds_PIXC.where(
        (lon >= bounds[0]) & (lon <= bounds[2]) &
        (lat  >= bounds[1]) & (lat  <= bounds[3]),
        drop=True
    )
    
    # Make mask
    mask = bitwiseMask(ds_subset, level)
    # mask = bitwiseMask(ds_PIXC, level)
    
    # Check that we didn't lose all pixels
    if mask.shape[0] == 0:
        print('This granule has no pixels after masking, exiting.')
        return None, None, None, None, None, None, None, None
    else:
        variables = ['cross_track', 'water_frac',
                     'pixel_area', 'height', 'geoid', 'solid_earth_tide',
                     'load_tide_fes', 'pole_tide', 'prior_water_prob', 
                     'phase_noise_std', 'dheight_dphase',
                     'classification_qual', 'geolocation_qual', 'sig0_qual',
                     'classification']
        
    # Convert PIXC to GeoDataFrame
    # gdf_PIXC = makeGDF(ds=ds_subset, mask=mask, data_vars=variables)
    gdf_PIXC = makeGDF(ds=ds_subset, mask=mask, data_vars=variables)

    # if gdf_PIXC.shape[0] == 0:
    #     print(f'WSE: {gdf_PIXC['wse']}')
    #     print(f'phase_noise_std**3: {gdf_PIXC['phase_noise_std']**3}')
    # Calculate wse
    gdf_PIXC.loc[:, 'wse'] = (gdf_PIXC.height
                              - gdf_PIXC.geoid
                              - gdf_PIXC.solid_earth_tide
                              - gdf_PIXC.load_tide_fes
                              - gdf_PIXC.pole_tide)
    gdf_PIXC = gdf_PIXC.drop(columns=['height', 'geoid', 'solid_earth_tide',
                                      'load_tide_fes', 'pole_tide'])
    
    # gdf_PIXC = gdf_PIXC.drop(columns=['latitude', 'longitude'])
    
    # # gdf_PIXC = gdf_PIXC.clip(buffered_mask)
    # gdf_PIXC = gdf_PIXC.clip(reach_mask)

    # ## Make indicator for wse aggregation
    # # (don't use land_near_water or dark_water)
    # wse_klass = [3.0, 4.0, 6.0, 7.0]
    
    # gdf_PIXC['agg_wse'] = 0
    # gdf_PIXC.loc[gdf_PIXC['klass'].isin(wse_klass), 'agg_wse'] = 1

    # gdf_PIXC = gdf_PIXC[gdf_PIXC['agg_wse'] == 1]

    return gdf_PIXC

### Run

In [ ]:
site_id = '12119000'

In [ ]:
nhd_gauges = gpd.read_parquet('/nas/cee-water/cjgleason/fiona/USGS_gauges/usgs_stations_with_stage_and_nhd_metadata.parquet')

In [ ]:
final = pd.read_parquet(
    f'/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_output/PIXC_vD_wse_2025_12_06/site_{site_id}.parquet')

In [ ]:
final.sort_values(by='n_pixels')

In [ ]:
# Extract reach mask and buffered reach masks
reach_mask, buffered_mask, bounds, huc2 = read_in_masks(site_id)

In [ ]:
# Find the correct SWOT tiles
pixc_files, lat, long, alt, vert_datum = find_SWOT_granules(site_id)

In [ ]:
search_string = '20250527'

for index, item in enumerate(pixc_files):
    if search_string in item:
        print(f"'{search_string}' found in '{item}' at index {index}")
        break
else:
    print(f"'{search_string}' not found in any list entry.")

In [ ]:
pixc_path, granule_name, pass_num, pixc_dt = get_metadata(
    pixc_files=pixc_files, index=38
)

In [ ]:
gdf_PIXC = read_PIXC_and_process(pixc_path=pixc_path,
                                 bounds=bounds,
                                 level=16)

In [ ]:
gdf_PIXC.klass.unique()

### Plot

In [ ]:
codes ={
    1: 'land',
    2: 'land_near_water',
    3: 'water_near_land',
    4: 'open_water',
    5: 'dark_water',
    6: 'low_coh_water_near_land',
    7: 'open_low_coh_water'
}

In [ ]:
palette = {2: 'darkolivegreen', 3: 'cornflowerblue', 4: 'blue', 5: 'hotpink',
           6: 'darkkhaki', 7: 'springgreen'}

In [ ]:
# Convert CRS to WGS 84 / UTM zone 18N
gdf_PIXC = gdf_PIXC.to_crs(epsg='32618')
reach_mask = reach_mask.to_crs(epsg='32618')

In [ ]:
reach_mask

In [ ]:
gauge = nhd_gauges[nhd_gauges['site_id'] == site_id].to_crs(epsg='32618')

In [ ]:
save_dir = '/nas/cee-water/cjgleason/fiona/for_presentations/2025_AGU/intro_figures/'

In [ ]:
bounds = [650535, 4729050, 651025, 4729330]
gdf_PIXC_c = gdf_PIXC#.clip(mask=bounds)

nums = list(np.unique(gdf_PIXC_c.klass).astype('int'))
labels = [codes[x] for x in nums]

# Make cmap
cmap = colors.ListedColormap([palette[x] for x in nums])

fig, ax = plt.subplots(figsize=(5,3))
reach_mask.plot(ax=ax, edgecolor='k', facecolor='none', linewidth=2)
gdf_PIXC_c.plot(ax=ax, alpha=0.9, column=gdf_PIXC_c.klass,
              categorical=True, cmap=cmap, s=10)
gauge.plot(ax=ax, color='k')

# Basemap
ctx.add_basemap(ax, crs=gdf_PIXC_c.crs,
                source=ctx.providers.CartoDB.Positron, alpha=0.9)

plt.xlim(650535, 651025)
plt.ylim(4729050, 4729330);

plt.axis('off');
#
# plt.savefig(f'{save_dir}hoosic.png', dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(save_dir, 'cedar.svg'), format='svg', bbox_inches='tight')

In [ ]:
gdf_PIXC_cor = gpd.sjoin(
    gdf_PIXC, reach_mask, how='inner',
    predicate='within').reset_index().drop(columns=['index', 'index_right']
)

In [ ]:
33.48-32.79

In [ ]:
33.81-.69

In [ ]:
from shapely.geometry import box

In [ ]:
651025 - 650535

In [ ]:
bounds = [650535, 4729050, 651025, 4729330]

# Invisible bounding polygon
bbox_poly = gpd.GeoDataFrame(
    geometry=[box(bounds[0], bounds[1], bounds[2], bounds[3])], 
    crs=gdf_PIXC_cor.crs
)

nums = list(np.unique(gdf_PIXC_c.klass).astype('int'))
labels = [codes[x] for x in nums]

# Make cmap
cmap = colors.ListedColormap([palette[x] for x in nums])

fig, ax = plt.subplots(figsize=(5,3))
reach_mask.plot(ax=ax, edgecolor='k', facecolor='none', linewidth=2)

# Plot invisible bounding box BEFORE add_basemap
bbox_poly.plot(ax=ax, alpha=0)  # invisible but sets extent

gdf_PIXC_cor.plot(ax=ax, alpha=0.9, column=gdf_PIXC_cor.klass,
              categorical=True, cmap=cmap, s=10)
gauge.plot(ax=ax, color='k')

# Basemap
ctx.add_basemap(ax, crs=gdf_PIXC_cor.crs,
                source=ctx.providers.CartoDB.Positron, alpha=0.9)

plt.xlim(650535, 651025)
plt.ylim(4729050, 4729330);

plt.axis('off');

# plt.savefig(f'{save_dir}hoosic_clipped.png', dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(save_dir, 'cedar_clipped.svg'), format='svg', bbox_inches='tight')